# Resume corrected C1 codec calibration
Run cells in order after attaching the ASVspoof dataset and **extracted latest backup**. Edit RESUME_FROM to its calibration directory. This notebook reuses corrected clean scores and preserves all original scores. It does not run old Parts A/B/C/D or publish active thresholds. Full parity and repository integration remain pending.


In [ ]:
# ── CONFIG — edit only this cell ─────────────────────────────────────────────
REPO_URL    = "https://github.com/Remigaraki/TwoStream-Audio-Forensics.git"
BRANCH      = "main"
DATA_ROOT   = "/kaggle/input/datasets/ulianazb/asvspoof-2024"   # contains flac_T/, flac_D/, ASVspoof5.*.metadata.txt

OPERATING_POINT = "eer"      # "eer" = equal error rates; "min_dcf" = paper's cost (bona fide wrongly flagged costs 1.9x)
MODELS      = ["C1", "A1"]
CODECS      = ["opus_16", "opus_32", "opus_64", "mp3_64", "mp3_128", "aac_128"]   # Part B; trim to split across sessions
RUN_PART_B  = True
PARITY_N    = 200            # Part C: number of clean test clips per model

RESUME_FROM = "/kaggle/input/datasets/rafdrilo/calibration-dataset-4/calibration"             # e.g. "/kaggle/input/<previous-version-output>/calibration" to reuse cached scores

# Paths (normally left alone)
REPO      = "/kaggle/tmp/TwoStream-Audio-Forensics"   # outside /kaggle/working: checkpoints are not re-saved as output
MANIFEST  = "/kaggle/working/manifest.csv"
CACHE_DIR = "/kaggle/working/calibration"
CODEC_VAL = "/kaggle/tmp/codec_val"                   # temporary; each codec folder is deleted once scored
UPLOAD_TO_GITHUB = False
PARITY_THRESHOLD_SHA256 = None
AUTO_UPLOAD_BACKUPS = True  # backup ZIPs only; GH_TOKEN must be enabled


In [ ]:
# ── Helpers: run a command, stream its output, stop on failure ───────────────
import os, shutil, subprocess, sys

ENV = {**os.environ, "PYTHONIOENCODING": "utf-8", "PYTHONUNBUFFERED": "1"}

def run(cmd, cwd=None):
    print("$", " ".join(cmd), flush=True)
    p = subprocess.Popen(cmd, cwd=cwd or REPO, env=ENV, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        print(line, end="", flush=True)
    if p.wait() != 0:
        raise RuntimeError(f"command failed with exit code {p.returncode}")

In [ ]:
# ── 1. Clone the repo and fetch ONLY the two served checkpoints (Git LFS) ────
os.makedirs(os.path.dirname(REPO), exist_ok=True)
if not os.path.isdir(os.path.join(REPO, ".git")):
    if os.path.exists(REPO):
        raise RuntimeError("Existing non-checkout directory; inspect it before cloning")
    # Skip LFS on clone so the other LFS checkpoints are not downloaded.
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, REPO_URL, REPO],
                   env={**ENV, "GIT_LFS_SKIP_SMUDGE": "1"}, check=True)
else:
    print("Reusing existing checkout without pulling over session changes")

if subprocess.run(["git", "lfs", "version"], capture_output=True).returncode != 0:
    run(["bash", "-c", "apt-get -qq update && apt-get -qq install -y git-lfs"], cwd="/")
run(["git", "lfs", "install", "--local"])
run(["git", "lfs", "pull", "--include",
     "checkpoints/setup_c/best_c1_attention.pt,checkpoints/setup_a/best_a_ep13_eer0105_0713_0401.pt"])
run(["git", "log", "-1", "--oneline"])

In [ ]:
# ── 2. Dependencies: keep Kaggle's GPU torch; match the PCA's scikit-learn ───
# pca.pkl was pickled by scikit-learn 1.8.0. pydub is needed only because src/train.py imports it.
run([sys.executable, "-m", "pip", "install", "-q", "scikit-learn==1.8.0", "pydub"])
run(["ffmpeg", "-hide_banner", "-version"])

In [ ]:
# ── 3. Environment + artifact check (hashes, LFS pointers, strict load) ──────
run([sys.executable, "-c", """
import sklearn, torch, torchaudio, soundfile
assert sklearn.__version__ == '1.8.0', sklearn.__version__
print('torch', torch.__version__, '| torchaudio', torchaudio.__version__, '| sklearn', sklearn.__version__,
      '| libsndfile', soundfile.__libsndfile_version__, '| cuda', torch.cuda.is_available())
from demo import inference as inf
for m in inf.MODELS:
    inf.load_model(m); print(m, 'verified and loaded')
"""])

In [ ]:
# Build/verify current-session paths; do not restore stale manifest paths.
if not os.path.isfile(MANIFEST):
    run([sys.executable, "scripts/build_manifest.py", "--input_root", DATA_ROOT, "--output", MANIFEST])
run([sys.executable, "scripts/verify_manifest.py", "--manifest", MANIFEST])


In [ ]:
# Embedded safeguards: no repository push required.
"""Dependency-free safeguards for resuming corrected Kaggle calibration."""
import csv
import hashlib
import json
import math
import shutil
import zipfile
from datetime import datetime, timezone
from pathlib import Path


def sha256(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()


def restore_tree(source, destination):
    source, destination = Path(source), Path(destination)
    if not source.is_dir():
        raise ValueError(f'Missing backup calibration tree: {source}')
    files = sorted(p for p in source.rglob('*') if p.is_file())
    # Preflight the entire tree before writing anything.
    for src in files:
        dst = destination / src.relative_to(source)
        if src.is_symlink() or not src.resolve().is_relative_to(source.resolve()):
            raise ValueError(f'Unsafe backup path: {src}')
        if dst.exists() and (not dst.is_file() or sha256(src) != sha256(dst)):
            raise ValueError(f'Restore conflict; preserve both versions and inspect: {dst}')
    for src in files:
        dst = destination / src.relative_to(source)
        if not dst.exists():
            dst.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(src, dst)


def validate_scores(path, expected):
    with Path(path).open(encoding='utf-8') as f:
        rows = list(csv.DictReader(f))
    ids = [r['utterance_id'] for r in rows]
    if len(ids) != len(expected) or len(set(ids)) != len(ids) or set(ids) != set(expected):
        raise ValueError(f'Incomplete or duplicate validation IDs: {path}')
    for row in rows:
        score = float(row['score'])
        if int(row['true_label']) != expected[row['utterance_id']] or not math.isfinite(score) or not 0 <= score <= 1:
            raise ValueError(f'Invalid score or label: {path}, {row["utterance_id"]}')


def backup(cache, thresholds, manifest, output_dir):
    stamp = datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S_%f')
    output = Path(output_dir) / f'calibration_backup_{stamp}.zip'
    partial = output.with_suffix('.partial')
    with zipfile.ZipFile(partial, 'w', zipfile.ZIP_DEFLATED) as z:
        z.write(thresholds, 'demo_thresholds.json')
        z.write(manifest, 'manifest.csv')
        for path in sorted(Path(cache).rglob('*')):
            if path.is_file():
                z.write(path, 'calibration/' + path.relative_to(cache).as_posix())
        z.writestr('backup_status.json', json.dumps({
            'created_utc': stamp, 'parity_status': 'unverified',
            'note': 'Corrected codec calibration progress; publication disabled.'
        }, indent=2))
    with zipfile.ZipFile(partial) as z:
        if z.testzip() is not None:
            raise ValueError('Backup ZIP integrity failure')
    partial.replace(output)
    print(f'Backup saved: {output} SHA256={sha256(output)}', flush=True)
    return output


In [ ]:
UPLOAD_TO_GITHUB = False
PARITY_THRESHOLD_SHA256 = None
restore_tree(RESUME_FROM, CACHE_DIR)
cache = Path(CACHE_DIR)
check = json.loads((cache / "batch_independence_check.json").read_text())
patch = cache / "lfcc_corrected.py"
expected_hash = check["lfcc_sha256"]
if not check["passed"] or sha256(patch) != expected_hash:
    raise RuntimeError("Saved LFCC patch/report mismatch")
CORRECTED_CACHE = cache / f"c1_corrected_{expected_hash[:12]}"
provenance = json.loads((CORRECTED_CACHE / "provenance.json").read_text())
if provenance["lfcc_sha256"] != expected_hash or sha256(CORRECTED_CACHE / "lfcc.py") != expected_hash:
    raise RuntimeError("Corrected cache preprocessing identity mismatch")
lfcc_path = Path(REPO) / "src/stream2/lfcc.py"
if sha256(lfcc_path) != expected_hash:
    saved_original = lfcc_path.with_suffix(".py.before_batch_fix")
    if saved_original.exists() and sha256(saved_original) != sha256(lfcc_path):
        raise RuntimeError("Existing LFCC backup differs; inspect before overwriting")
    if not saved_original.exists():
        shutil.copy2(lfcc_path, saved_original)
    shutil.copy2(patch, lfcc_path)
assert sha256(lfcc_path) == expected_hash
with open(MANIFEST, encoding="utf-8") as f:
    expected_labels = {r["utterance_id"]: int(r["label"]) for r in csv.DictReader(f) if r["split"] == "val"}
if len(expected_labels) != 18769:
    raise RuntimeError("Unexpected validation coverage")
checkpoint_prefix = "3047175ddc1b"
clean = CORRECTED_CACHE / f"val_C1_{checkpoint_prefix}_clean.csv"
validate_scores(clean, expected_labels)
for score_file in CORRECTED_CACHE.glob("val_C1_*.csv"):
    validate_scores(score_file, expected_labels)
thresholds_path = Path(REPO) / "demo/thresholds.json"
# Restore the staged calibration snapshot, never the original C1 codec summaries.
snapshot = CORRECTED_CACHE / "thresholds_with_codec_checks.json"
if not snapshot.exists():
    snapshot = CORRECTED_CACHE / "thresholds_after_clean_recalibration.json"
staged = json.loads(snapshot.read_text())
if staged["C1"]["provenance"]["checkpoint_sha256"] != "3047175ddc1b2f35ed7f31541c24870f3adffdf5099a0f5dc257f651dc3a930d":
    raise RuntimeError("Checkpoint identity mismatch")
before_resume = CORRECTED_CACHE / "thresholds_before_resume.json"
if thresholds_path.exists() and not before_resume.exists():
    shutil.copy2(thresholds_path, before_resume)
shutil.copy2(snapshot, thresholds_path)
clean_threshold = staged["C1"]["threshold"]
print("Restored corrected clean calibration; original caches preserved. Parity remains unverified.")


In [ ]:
import base64
import hashlib
from pathlib import Path
from urllib.parse import quote

import requests
from kaggle_secrets import UserSecretsClient


def upload_calibration_backup(zip_file):
    path = Path(zip_file)
    if not path.is_file():
        raise FileNotFoundError("Run the ZIP-backup cell first.")

    # Conservative size limit for storing backups directly in Git.
    if path.stat().st_size > 25 * 1024 * 1024:
        raise ValueError("Backup exceeds 25 MiB; use a GitHub Release asset instead.")

    data = path.read_bytes()
    blob_sha = hashlib.sha1(
        f"blob {len(data)}\0".encode() + data
    ).hexdigest()

    repo = "Remigaraki/TwoStream-Audio-Forensics"
    branch = BRANCH
    remote_path = f"calibration_backups/{path.name}"
    url = (
        f"https://api.github.com/repos/{repo}/contents/"
        f"{quote(remote_path, safe='/')}"
    )

    try:
        token = UserSecretsClient().get_secret("GH_TOKEN")
    except Exception:
        raise RuntimeError(
            "Enable GH_TOKEN for this notebook in Kaggle Secrets."
        ) from None

    with requests.Session() as session:
        session.headers.update({
            "Authorization": f"Bearer {token}",
            "Accept": "application/vnd.github+json",
            "X-GitHub-Api-Version": "2022-11-28",
        })

        # A rerun should not overwrite an existing, different backup.
        existing = session.get(
            url, params={"ref": branch}, timeout=60
        )
        if existing.status_code == 200:
            info = existing.json()
            if info["sha"] == blob_sha:
                print("Backup already saved:", info["html_url"])
                return
            raise RuntimeError(
                "A different backup has this filename. "
                "Rerun the ZIP cell to generate a new timestamp."
            )
        if existing.status_code != 404:
            raise RuntimeError(
                f"GitHub lookup failed: HTTP {existing.status_code}. "
                "Check repository access and token permissions."
            )

        response = session.put(
            url,
            json={
                "message": f"Save Kaggle calibration backup: {path.name}",
                "branch": branch,
                "content": base64.b64encode(data).decode("ascii"),
            },
            timeout=180,
        )
        if response.status_code != 201:
            raise RuntimeError(
                f"GitHub upload failed: HTTP {response.status_code}. "
                "Check Contents write permission and branch protection. "
                "The local ZIP is still available."
            )

        result = response.json()
        if result["content"]["sha"] != blob_sha:
            raise RuntimeError("Uploaded file hash does not match.")

        print("Backup uploaded and hash verified:")
        print(result["content"]["html_url"])



def save_progress():
    output = backup(CACHE_DIR, thresholds_path, MANIFEST, "/kaggle/working")
    if AUTO_UPLOAD_BACKUPS:
        upload_calibration_backup(output)
    return output
# Protect restored state before expensive work.
save_progress()


In [ ]:
conditions = ["opus_16", "opus_32", "opus_64", "mp3_64", "mp3_128", "aac_128"]
for condition in conditions:
    if sha256(lfcc_path) != expected_hash:
        raise RuntimeError("LFCC changed during calibration")
    score_file = CORRECTED_CACHE / f"val_C1_{checkpoint_prefix}_{condition}.csv"
    # A separate root exposes only this condition to the existing calibrator.
    # Existing corrected caches are still included in accumulated summaries.
    isolated_root = Path(CODEC_VAL) / "corrected_c1" / condition
    if not score_file.exists():
        run([sys.executable, "scripts/make_codec_testsets.py", "--manifest", MANIFEST,
             "--output_base", str(isolated_root), "--split", "val", "--conditions", condition])
        condition_dir = isolated_root / condition
        with open(MANIFEST, encoding="utf-8") as f:
            missing = [r["utterance_id"] for r in csv.DictReader(f)
                       if r["split"] == "val" and
                       (not (condition_dir / Path(r["file_path"]).name).is_file()
                        or (condition_dir / Path(r["file_path"]).name).stat().st_size == 0)]
        if missing:
            raise RuntimeError(f"Incomplete codec audio: {len(missing)} missing/empty files")
    else:
        validate_scores(score_file, expected_labels)
        print(f"[{condition}] reusing corrected scores")
    run([sys.executable, "demo/calibrate_threshold.py", "--model", "C1", "--manifest", MANIFEST,
         "--operating_point", OPERATING_POINT, "--cache_dir", str(CORRECTED_CACHE),
         "--codec_root", str(isolated_root), "--num_workers", "4"])
    validate_scores(score_file, expected_labels)
    result = json.loads(thresholds_path.read_text())
    if result["C1"]["threshold"] != clean_threshold:
        raise RuntimeError("Clean threshold changed unexpectedly")
    if condition not in (result["C1"]["provenance"]["codec_check_val"] or {}):
        raise RuntimeError("Requested codec summary missing")
    shutil.copy2(thresholds_path, CORRECTED_CACHE / "thresholds_with_codec_checks.json")
    progress = {"lfcc_sha256": expected_hash, "status": "codec_calibration_in_progress",
                "completed_conditions": [c for c in conditions if
                    (CORRECTED_CACHE / f"val_C1_{checkpoint_prefix}_{c}.csv").exists()],
                "parity_status": "unverified"}
    (CORRECTED_CACHE / "codec_progress.json").write_text(json.dumps(progress, indent=2))
    save_progress()  # failure stops here; local ZIP remains available
    print(f"[{condition}] saved and backed up")
# Report completion only after validating every corrected cache and summary.
for condition in conditions:
    validate_scores(CORRECTED_CACHE / f"val_C1_{checkpoint_prefix}_{condition}.csv", expected_labels)
result = json.loads(thresholds_path.read_text())
summaries = result["C1"]["provenance"]["codec_check_val"]
assert set(summaries) == set(conditions)
print(json.dumps(summaries, indent=2))
print("Corrected codec calibration complete. Full demo parity, local integration and deployment remain pending.")
